In [2]:
# 1. Strip away the problematic vision and audio dependencies causing the loop crash
!pip uninstall -y torchvision torchaudio

# 2. Force install clean, fully compatible text-processing frameworks
!pip install --no-cache-dir datasets==3.0.0 transformers==4.44.2 accelerate>=0.26.0

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.27 requires torchvision, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


# Data Initialization and Google Drive Loading

In [1]:
from google.colab import drive
import numpy as np
import scipy.sparse as sp
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive')

# Update this path if your folder name differs
DRIVE_PATH = "/content/drive/MyDrive/Natural Language Processing/Project"

print(f"\nChecking folder:\n{DRIVE_PATH}")

# Verify folder exists
if not os.path.exists(DRIVE_PATH):
    raise FileNotFoundError(
        f"Folder not found:\n{DRIVE_PATH}\n\n"
        "Check your Google Drive folder name and path."
    )

print("\nFiles found in folder:")
for file in sorted(os.listdir(DRIVE_PATH)):
    print(file)

print("\nLoading dataset arrays directly from Google Drive...")

# --------------------------
# Load Labels
# --------------------------

y_train_path = os.path.join(DRIVE_PATH, "y_train.npy")
y_val_path = os.path.join(DRIVE_PATH, "y_val.npy")
y_test_path = os.path.join(DRIVE_PATH, "y_test.npy")

if not os.path.exists(y_train_path):
    raise FileNotFoundError(f"Missing file: {y_train_path}")

if not os.path.exists(y_val_path):
    raise FileNotFoundError(f"Missing file: {y_val_path}")

if not os.path.exists(y_test_path):
    raise FileNotFoundError(f"Missing file: {y_test_path}")

y_train = np.load(y_train_path, allow_pickle=True)
y_val = np.load(y_val_path, allow_pickle=True)
y_test = np.load(y_test_path, allow_pickle=True)

# --------------------------
# Load TF-IDF Features
# --------------------------

X_train_tfidf = sp.load_npz(
    os.path.join(DRIVE_PATH, "X_train_tfidf.npz")
)

X_val_tfidf = sp.load_npz(
    os.path.join(DRIVE_PATH, "X_val_tfidf.npz")
)

X_test_tfidf = sp.load_npz(
    os.path.join(DRIVE_PATH, "X_test_tfidf.npz")
)

# --------------------------
# Load Word2Vec Features
# --------------------------

X_train_w2v = np.load(
    os.path.join(DRIVE_PATH, "X_train_w2v.npy")
)

X_val_w2v = np.load(
    os.path.join(DRIVE_PATH, "X_val_w2v.npy")
)

X_test_w2v = np.load(
    os.path.join(DRIVE_PATH, "X_test_w2v.npy")
)

print("\nAll features and labels loaded successfully!")

print(
    f"TF-IDF - "
    f"Train: {X_train_tfidf.shape}, "
    f"Val: {X_val_tfidf.shape}, "
    f"Test: {X_test_tfidf.shape}"
)

print(
    f"Word2Vec - "
    f"Train: {X_train_w2v.shape}, "
    f"Val: {X_val_w2v.shape}, "
    f"Test: {X_test_w2v.shape}"
)

print(
    f"\nLabels - "
    f"Train: {y_train.shape}, "
    f"Val: {y_val.shape}, "
    f"Test: {y_test.shape}"
)

Mounted at /content/drive

Checking folder:
/content/drive/MyDrive/Natural Language Processing/Project

Files found in folder:
X_test_tfidf.npz
X_test_w2v.npy
X_train_tfidf.npz
X_train_w2v.npy
X_val_tfidf.npz
X_val_w2v.npy
best_confusion_matrix.npy
best_emotion_model.pkl
best_model
best_model_classification_report.csv
emotion_labels.npy
model_comparison_results.csv
test_cleaned.csv
train_cleaned.csv
val_cleaned.csv
y_test.npy
y_train.npy
y_val.npy

Loading dataset arrays directly from Google Drive...

All features and labels loaded successfully!
TF-IDF - Train: (16000, 5000), Val: (2000, 5000), Test: (2000, 5000)
Word2Vec - Train: (16000, 300), Val: (2000, 300), Test: (2000, 300)

Labels - Train: (16000,), Val: (2000,), Test: (2000,)


# Modeling Framework & Evaluation Metrics Setup

In [2]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

# Global dictionary to collect all experiment scores
results_registry = {}

def train_and_evaluate(model, X_train, y_train, X_val, y_val, X_test, y_test, experiment_name):
    print(f"==========================================================")
    print(f"Training & Evaluating Pipeline: {experiment_name}...")
    print(f"==========================================================")

    # Train the algorithm
    model.fit(X_train, y_train)

    # 1. Validation Set Evaluation (Used strictly for selecting the winning model configuration)
    val_preds = model.predict(X_val)
    val_acc = accuracy_score(y_val, val_preds)
    _, _, val_f1, _ = precision_recall_fscore_support(y_val, val_preds, average='weighted', zero_division=0)
    print(f"Validation -> Accuracy: {val_acc:.4f} | Weighted F1-Score: {val_f1:.4f}")

    # 2. Final Test Set Evaluation (Held out entirely, evaluated for final documentation reporting)
    test_predictions = model.predict(X_test)
    test_acc = accuracy_score(y_test, test_predictions)
    test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(y_test, test_predictions, average='weighted', zero_division=0)
    test_cm = confusion_matrix(y_test, test_predictions)

    # Generate the dictionary format of the classification report for file export later
    test_report_dict = classification_report(y_test, test_predictions, output_dict=True, zero_division=0)

    # Print Comprehensive Per-Class Classification Report to screen
    print("\nDetailed Test Classification Report:")
    print(classification_report(y_test, test_predictions, zero_division=0))

    # Save statistics including both validation tracking metrics and test verification matrices
    results_registry[experiment_name] = {
        'model_instance': model,
        'val_f1_score': val_f1,          # <-- Key addition: used for choosing the model
        'test_accuracy': test_acc,
        'test_precision': test_prec,
        'test_recall': test_rec,
        'test_f1_score': test_f1,
        'test_confusion_matrix': test_cm,
        'test_report_dict': test_report_dict
    }
    print(f"Final Test Performance -> Acc: {test_acc:.4f} | Weighted F1: {test_f1:.4f}\n")

# Complete Machine Learning Benchmarking

In [3]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# --- Experiment 1: Multinomial Naive Bayes + TF-IDF ---
train_and_evaluate(
    model=MultinomialNB(),
    X_train=X_train_tfidf, y_train=y_train,
    X_val=X_val_tfidf, y_val=y_val,
    X_test=X_test_tfidf, y_test=y_test,
    experiment_name="Naive_Bayes_TFIDF"
)

# --- Experiment 2: Logistic Regression + TF-IDF (Highly reliable, fast, supports predict_proba) ---
train_and_evaluate(
    model=LogisticRegression(max_iter=2000, random_state=42),
    X_train=X_train_tfidf, y_train=y_train,
    X_val=X_val_tfidf, y_val=y_val,
    X_test=X_test_tfidf, y_test=y_test,
    experiment_name="LogisticReg_TFIDF"
)

# --- Experiment 3: Logistic Regression + Word2Vec ---
train_and_evaluate(
    model=LogisticRegression(max_iter=2000, random_state=42),
    X_train=X_train_w2v, y_train=y_train,
    X_val=X_val_w2v, y_val=y_val,
    X_test=X_test_w2v, y_test=y_test,
    experiment_name="LogisticReg_Word2Vec"
)

# --- Experiment 4: Support Vector Machine (Probability Ready) + Word2Vec ---
train_and_evaluate(
    model=SVC(kernel='linear', probability=True, random_state=42),
    X_train=X_train_w2v, y_train=y_train,
    X_val=X_val_w2v, y_val=y_val,
    X_test=X_test_w2v, y_test=y_test,
    experiment_name="SVM_Word2Vec"
)

Training & Evaluating Pipeline: Naive_Bayes_TFIDF...
Validation -> Accuracy: 0.8140 | Weighted F1-Score: 0.7950

Detailed Test Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.94      0.87       581
           1       0.75      0.98      0.85       695
           2       1.00      0.30      0.46       159
           3       0.94      0.63      0.76       275
           4       0.86      0.66      0.74       224
           5       1.00      0.12      0.22        66

    accuracy                           0.80      2000
   macro avg       0.89      0.61      0.65      2000
weighted avg       0.83      0.80      0.78      2000

Final Test Performance -> Acc: 0.8035 | Weighted F1: 0.7796

Training & Evaluating Pipeline: LogisticReg_TFIDF...
Validation -> Accuracy: 0.8920 | Weighted F1-Score: 0.8893

Detailed Test Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.

# Deep Learning Experiment: RoBERTa Fine-Tuning

In [5]:
import torch
import numpy as np
import pandas as pd
import os

from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("Loading original text datasets...")
train_df = pd.read_csv(os.path.join(DRIVE_PATH, 'train_cleaned.csv'))
val_df = pd.read_csv(os.path.join(DRIVE_PATH, 'val_cleaned.csv'))
test_df = pd.read_csv(os.path.join(DRIVE_PATH, 'test_cleaned.csv'))

# Extract text + labels
X_train_text = train_df['text'].astype(str).tolist()
X_val_text = val_df['text'].astype(str).tolist()
X_test_text = test_df['text'].astype(str).tolist()

y_train_roberta = train_df['label'].tolist()
y_val_roberta = val_df['label'].tolist()
y_test_roberta = test_df['label'].tolist()

print("Tokenizer initializing...")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Tokenization function
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Create HuggingFace datasets
train_dataset = Dataset.from_dict({"text": X_train_text, "label": y_train_roberta})
val_dataset   = Dataset.from_dict({"text": X_val_text, "label": y_val_roberta})
test_dataset  = Dataset.from_dict({"text": X_test_text, "label": y_test_roberta})

# Tokenize datasets
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

# Set torch format
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print("Loading RoBERTa pre-trained weights architecture...")
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(set(y_train_roberta))
)

# Metrics evaluation module
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# Training arguments (Fixed evaluation_strategy bug)
training_args = TrainingArguments(
    output_dir="./roberta_results",
    eval_strategy="epoch",            # <-- Fixed structural bug
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    report_to="none"
)

# Trainer (Fixed processing_class object assignment framework)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

print("Fine-Tuning RoBERTa on GPU pipeline...")
trainer.train()

# Validation evaluation
print("\nEvaluating Validation Set...")
val_results = trainer.evaluate(val_dataset)
val_f1 = val_results["eval_f1"]
print(f"Validation F1 = {val_f1:.4f}")

# Test evaluation
print("\nEvaluating Locked Test Set...")
predictions = trainer.predict(test_dataset)
test_preds = np.argmax(predictions.predictions, axis=1)

test_acc = accuracy_score(y_test_roberta, test_preds)
test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(
    y_test_roberta, test_preds, average="weighted", zero_division=0
)
test_cm = confusion_matrix(y_test_roberta, test_preds)
test_report_dict = classification_report(
    y_test_roberta, test_preds, output_dict=True, zero_division=0
)

print("\nDetailed Test Classification Report (RoBERTa):")
print(classification_report(y_test_roberta, test_preds, zero_division=0))

# Save statistics safely to main collection array
results_registry["RoBERTa"] = {
    "model_instance": trainer,        # Target the trainer configuration for safe transformer handles
    "val_f1_score": val_f1,
    "test_accuracy": test_acc,
    "test_precision": test_prec,
    "test_recall": test_rec,
    "test_f1_score": test_f1,
    "test_confusion_matrix": test_cm,
    "test_report_dict": test_report_dict
}

Loading original text datasets...
Tokenizer initializing...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

...

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fine-Tuning RoBERTa on GPU pipeline...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.266800,0.213498,0.930000,0.932099,0.930000,0.929990
2,0.140500,0.155829,0.940000,0.941992,0.940000,0.939346
3,0.093000,0.151766,0.935000,0.935601,0.935000,0.934816



Evaluating Validation Set...


Validation F1 = 0.9393

Evaluating Locked Test Set...

Detailed Test Classification Report (RoBERTa):
              precision    recall  f1-score   support

           0       0.97      0.96      0.96       581
           1       0.92      0.97      0.95       695
           2       0.96      0.69      0.80       159
           3       0.93      0.94      0.94       275
           4       0.91      0.86      0.88       224
           5       0.67      0.91      0.77        66

    accuracy                           0.93      2000
   macro avg       0.89      0.89      0.88      2000
weighted avg       0.93      0.93      0.93      2000



# Academic Selection, Report Generation & Export

In [6]:
import joblib

# 1. Compile results into a clean, comprehensive evaluation table
summary_data = []
for setup, metrics in results_registry.items():
    summary_data.append({
        'Model Pipeline Configuration': setup,
        'Validation F1-Score': metrics['val_f1_score'],
        'Test Accuracy': metrics['test_accuracy'],
        'Test Precision': metrics['test_precision'],
        'Test Recall': metrics['test_recall'],
        'Test F1-Score': metrics['test_f1_score']
    })

df_performance = pd.DataFrame(summary_data)

# 2. PROPER ACADEMIC SELECTION: Choose winner strictly based on Validation F1
winning_config = df_performance.loc[df_performance['Validation F1-Score'].idxmax()]['Model Pipeline Configuration']

winning_entry = results_registry[winning_config]
winning_cm = winning_entry['test_confusion_matrix']
winning_report_dict = winning_entry['test_report_dict']

print("==================================== FINAL MODEL PERFORMANCE COMPARISON ====================================")
print(df_performance.to_string(index=False))
print(f"\nWinning Model Configuration Selected (via Validation Tuning): {winning_config}")

# 3. Export class labels for Member 4's axes
emotion_labels = np.unique(y_test_roberta)

# 4. Save clean, secure assets back to Google Drive (with Deep Learning safety check)
if winning_config == "RoBERTa":
    roberta_save_path = os.path.join(DRIVE_PATH, 'roberta_production_model')
    winning_entry['model_instance'].save_model(roberta_save_path)
    print(f"RoBERTa Transformer saved via native API to directory: {roberta_save_path}")
else:
    joblib.dump(winning_entry['model_instance'], os.path.join(DRIVE_PATH, 'best_emotion_model.pkl'))
    print(f"Classical ML model '{winning_config}' saved as 'best_emotion_model.pkl'.")

# Save info metadata file for clear team communication
with open(os.path.join(DRIVE_PATH, "best_model_info.txt"), "w") as f:
    f.write(winning_config)

# Export comparative metrics spreadsheet
df_performance.to_csv(os.path.join(DRIVE_PATH, 'model_comparison_results.csv'), index=False)

# Export the detailed per-class stats matrix
df_class_report = pd.DataFrame(winning_report_dict).transpose()
df_class_report.to_csv(os.path.join(DRIVE_PATH, 'best_model_classification_report.csv'), index=True)

# Export structural arrays for heatmap design
np.save(os.path.join(DRIVE_PATH, 'best_confusion_matrix.npy'), winning_cm)
np.save(os.path.join(DRIVE_PATH, 'emotion_labels.npy'), emotion_labels)

print("All optimization data, heatmap metrics, and label vectors written to Google Drive successfully!")

==================================== FINAL MODEL PERFORMANCE COMPARISON ====================================
Model Pipeline Configuration  Validation F1-Score  Test Accuracy  Test Precision  Test Recall  Test F1-Score
           Naive_Bayes_TFIDF             0.795032         0.8035        0.833154       0.8035       0.779562
           LogisticReg_TFIDF             0.889304         0.8805        0.880260       0.8805       0.876788
        LogisticReg_Word2Vec             0.301996         0.4015        0.285085       0.4015       0.308779
                SVM_Word2Vec             0.293737         0.3960        0.249053       0.3960       0.297527
                     RoBERTa             0.939346         0.9265        0.930391       0.9265       0.925844

Winning Model Configuration Selected (via Validation Tuning): RoBERTa
RoBERTa Transformer saved via native API to directory: /content/drive/MyDrive/Natural Language Processing/Project/roberta_production_model
All optimization data, heat